# Understanding Pooling Operations in Convolutional Neural Networks

This notebook teaches pooling operations from first principles, building intuition for why they're essential in CNNs and how they enable translation invariance and efficient feature extraction.

**What you'll learn:**
1. What pooling is and why CNNs need it
2. MaxPooling and AveragePooling
3. Global pooling variants
4. How pooling affects spatial dimensions and receptive fields
5. Spatial invariance and translation robustness
6. When to use each pooling type

**Prerequisites:** Basic understanding of convolutions (see basics-tensors-convolution.ipynb)

## Setup

Let's import the necessary libraries and set up our environment for reproducibility.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {torch.cuda.is_available() and 'CUDA' or 'CPU'}")

## Part 1: What is Pooling?

**Pooling** (also called **downsampling** or **subsampling**) is an operation that reduces the spatial dimensions (height and width) of feature maps while retaining important information.

**Core idea:** Slide a window over the input and aggregate the values within each window into a single output value.

**Three key benefits:**
1. **Dimensionality reduction**: Reduces computation and memory usage
2. **Translation invariance**: Makes the network robust to small shifts in input
3. **Receptive field expansion**: Each layer sees a larger region of the original input

Let's start with a concrete example to understand the mechanics.

## Part 2: Max Pooling - The Most Common Operation

**Max Pooling** takes the maximum value within each window. It's the most popular pooling operation in modern CNNs.

**Example:** 2x2 max pooling with stride 2
```
Input (4x4):          Output (2x2):
[1  3  2  4]          [6  8]
[5  6  1  8]    →     [9  7]
[2  9  0  3]
[4  7  5  1]
```

The 2x2 windows are:
- Top-left: [1,3,5,6] → max = 6
- Top-right: [2,4,1,8] → max = 8
- Bottom-left: [2,9,4,7] → max = 9
- Bottom-right: [0,3,5,1] → max = 7

Let's implement this manually:

In [ ]:
# Create a simple 4x4 input
input_2d = torch.tensor([
    [1., 3., 2., 4.],
    [5., 6., 1., 8.],
    [2., 9., 0., 3.],
    [4., 7., 5., 1.]
])

print("Input (4x4):")
print(input_2d)

# Manual max pooling with 2x2 kernel, stride 2
pool_size = 2
stride = 2
output_size = input_2d.shape[0] // stride

output_manual = torch.zeros(output_size, output_size)

print("\nManual max pooling computation:")
for i in range(output_size):
    for j in range(output_size):
        window = input_2d[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
        max_val = window.max()
        output_manual[i, j] = max_val
        print(f"Position ({i},{j}): window=\n{window}\nmax={max_val:.1f}\n")

print("Output (2x2):")
print(output_manual)

### Max Pooling with PyTorch

PyTorch provides `nn.MaxPool2d` for max pooling. The input shape is `(batch, channels, height, width)`.

Let's verify our manual computation:

In [ ]:
# PyTorch max pooling
# Input shape: (batch=1, channels=1, height=4, width=4)
input_batch = input_2d.unsqueeze(0).unsqueeze(0)
print("Input shape:", input_batch.shape)

# Create MaxPool2d layer: kernel_size=2, stride=2
maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

# Apply max pooling
output_pytorch = maxpool(input_batch)
print("\nPyTorch output shape:", output_pytorch.shape)
print("PyTorch output:")
print(output_pytorch.squeeze())

print("\nMatches manual computation?", torch.allclose(output_pytorch.squeeze(), output_manual))

### Visualizing Max Pooling

Let's create a visual representation to see how max pooling selects the strongest activations:

In [ ]:
# Visualize max pooling operation
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Plot input
im1 = axes[0].imshow(input_2d, cmap='viridis', interpolation='nearest')
axes[0].set_title('Input (4x4)', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))
axes[0].grid(True, color='white', linewidth=2)
plt.colorbar(im1, ax=axes[0])

# Add text values
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{input_2d[i,j]:.0f}', 
                    ha='center', va='center', color='white', fontsize=12, fontweight='bold')

# Draw 2x2 pooling windows
for i in range(0, 4, 2):
    for j in range(0, 4, 2):
        rect = Rectangle((j-0.5, i-0.5), 2, 2, fill=False, edgecolor='red', linewidth=3)
        axes[0].add_patch(rect)

# Plot output
im2 = axes[1].imshow(output_manual, cmap='viridis', interpolation='nearest')
axes[1].set_title('Max Pooled Output (2x2)', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(2))
axes[1].set_yticks(range(2))
axes[1].grid(True, color='white', linewidth=2)
plt.colorbar(im2, ax=axes[1])

# Add text values
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{output_manual[i,j]:.0f}', 
                    ha='center', va='center', color='white', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Notice: Each output value is the maximum from its corresponding 2x2 window (shown in red).")

## Part 3: Average Pooling - The Smooth Alternative

**Average Pooling** takes the average (mean) value within each window instead of the maximum.

**When to use average pooling:**
- When you want to preserve overall feature presence rather than peak activations
- Often used in the final layers of CNNs (Global Average Pooling)
- Can provide smoother gradients during backpropagation

Let's apply average pooling to the same input:

In [ ]:
# Manual average pooling
output_avg_manual = torch.zeros(output_size, output_size)

print("Manual average pooling computation:")
for i in range(output_size):
    for j in range(output_size):
        window = input_2d[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
        avg_val = window.mean()
        output_avg_manual[i, j] = avg_val
        print(f"Position ({i},{j}): window=\n{window}\naverage={avg_val:.2f}\n")

print("Output (2x2):")
print(output_avg_manual)

### Average Pooling with PyTorch

PyTorch provides `nn.AvgPool2d` for average pooling:

In [ ]:
# PyTorch average pooling
avgpool = nn.AvgPool2d(kernel_size=2, stride=2)
output_avg_pytorch = avgpool(input_batch)

print("PyTorch average pooling output:")
print(output_avg_pytorch.squeeze())

print("\nMatches manual computation?", torch.allclose(output_avg_pytorch.squeeze(), output_avg_manual))

### Comparing Max Pooling vs Average Pooling

Let's visualize the difference side by side:

In [ ]:
# Compare max pooling vs average pooling
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Input
im1 = axes[0].imshow(input_2d, cmap='viridis', interpolation='nearest')
axes[0].set_title('Input (4x4)', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))
axes[0].grid(True, color='white', linewidth=2)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{input_2d[i,j]:.0f}', ha='center', va='center', 
                    color='white', fontsize=10, fontweight='bold')

# Max pooling output
im2 = axes[1].imshow(output_manual, cmap='viridis', interpolation='nearest')
axes[1].set_title('Max Pooling (2x2)', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(2))
axes[1].set_yticks(range(2))
axes[1].grid(True, color='white', linewidth=2)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{output_manual[i,j]:.0f}', ha='center', va='center', 
                    color='white', fontsize=10, fontweight='bold')

# Average pooling output
im3 = axes[2].imshow(output_avg_manual, cmap='viridis', interpolation='nearest')
axes[2].set_title('Average Pooling (2x2)', fontsize=14, fontweight='bold')
axes[2].set_xticks(range(2))
axes[2].set_yticks(range(2))
axes[2].grid(True, color='white', linewidth=2)
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, f'{output_avg_manual[i,j]:.1f}', ha='center', va='center', 
                    color='white', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey difference:")
print("- Max pooling selects the strongest activation (highest value)")
print("- Average pooling considers all activations equally")

## Part 4: Why Downsample?

Pooling provides three critical benefits for CNNs. Let's explore each one:

### 1. Computational Efficiency

Reducing spatial dimensions dramatically decreases:
- Number of parameters in subsequent layers
- Memory usage
- Computation time

Let's quantify this:

In [ ]:
# Demonstrate computational savings
# Scenario: Feature maps before and after pooling
feature_map_before = torch.randn(1, 64, 56, 56)  # 64 channels, 56x56 spatial
feature_map_after = torch.randn(1, 64, 28, 28)   # After 2x2 pooling with stride 2

# Calculate memory
memory_before = feature_map_before.numel() * 4 / 1024  # 4 bytes per float32, convert to KB
memory_after = feature_map_after.numel() * 4 / 1024

print(f"Feature map before pooling: {feature_map_before.shape}")
print(f"  Memory: {memory_before:.2f} KB")
print(f"  Total elements: {feature_map_before.numel():,}\n")

print(f"Feature map after pooling: {feature_map_after.shape}")
print(f"  Memory: {memory_after:.2f} KB")
print(f"  Total elements: {feature_map_after.numel():,}\n")

reduction = memory_before / memory_after
print(f"Memory reduction: {reduction:.1f}x")
print(f"Spatial dimension reduction: {56/28:.1f}x in each dimension")
print(f"Total spatial reduction: {(56*56)/(28*28):.1f}x")

### 2. Impact on Fully Connected Layers

Pooling is especially important before fully connected layers, where the number of parameters explodes with large spatial dimensions:

In [ ]:
# Compare parameter count with and without pooling
# Scenario: Flatten and connect to a layer with 512 units

# Without pooling: 64 channels × 56 × 56 = 200,704 inputs
input_size_no_pool = 64 * 56 * 56
fc_units = 512
params_no_pool = input_size_no_pool * fc_units

# With pooling: 64 channels × 28 × 28 = 50,176 inputs
input_size_with_pool = 64 * 28 * 28
params_with_pool = input_size_with_pool * fc_units

print("Scenario: Connecting CNN features to fully connected layer with 512 units\n")
print(f"Without pooling:")
print(f"  Input size: {input_size_no_pool:,}")
print(f"  Parameters: {params_no_pool:,}\n")

print(f"With 2x2 pooling (stride 2):")
print(f"  Input size: {input_size_with_pool:,}")
print(f"  Parameters: {params_with_pool:,}\n")

print(f"Parameter reduction: {params_no_pool/params_with_pool:.1f}x fewer parameters!")
print(f"Saved parameters: {(params_no_pool - params_with_pool):,}")

## Part 5: Spatial Invariance and Translation Robustness

**Spatial invariance** means the network's predictions are robust to small translations (shifts) of the input.

**Example:** A cat detector should recognize a cat whether it's in the top-left or slightly shifted to the right.

Pooling provides **local translation invariance** by aggregating nearby features. Let's demonstrate:

In [ ]:
# Demonstrate translation invariance with max pooling
# Create a simple pattern: a "peak" at different positions

# Pattern 1: Peak at position (1, 1)
pattern1 = torch.tensor([
    [0., 0., 0., 0.],
    [0., 9., 0., 0.],
    [0., 0., 0., 0.],
    [0., 0., 0., 0.]
])

# Pattern 2: Peak shifted to position (1, 2)
pattern2 = torch.tensor([
    [0., 0., 0., 0.],
    [0., 0., 9., 0.],
    [0., 0., 0., 0.],
    [0., 0., 0., 0.]
])

# Pattern 3: Peak shifted to position (2, 1)
pattern3 = torch.tensor([
    [0., 0., 0., 0.],
    [0., 0., 0., 0.],
    [0., 9., 0., 0.],
    [0., 0., 0., 0.]
])

# Apply max pooling to all patterns
maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

output1 = maxpool(pattern1.unsqueeze(0).unsqueeze(0))
output2 = maxpool(pattern2.unsqueeze(0).unsqueeze(0))
output3 = maxpool(pattern3.unsqueeze(0).unsqueeze(0))

print("Pattern 1 (peak at 1,1):")
print(pattern1)
print("After max pooling:")
print(output1.squeeze())
print()

print("Pattern 2 (peak at 1,2):")
print(pattern2)
print("After max pooling:")
print(output2.squeeze())
print()

print("Pattern 3 (peak at 2,1):")
print(pattern3)
print("After max pooling:")
print(output3.squeeze())
print()

print("Notice: Patterns 1 and 2 produce the SAME pooled output!")
print("The network is invariant to small horizontal shifts.")

### Visualizing Translation Invariance

Let's create a more realistic example with a simple "edge" pattern:

In [ ]:
# Create edge patterns at different positions
def create_vertical_edge(shift=0):
    """Create a 6x6 image with a vertical edge, optionally shifted"""
    img = torch.zeros(6, 6)
    img[:, :3+shift] = 1.0
    return img

edge_0 = create_vertical_edge(0)
edge_1 = create_vertical_edge(1)
edge_2 = create_vertical_edge(2)

# Apply convolution (edge detector) + max pooling
edge_kernel = torch.tensor([[-1., 1.]]).repeat(1, 1, 3, 1)  # Vertical edge detector
conv = nn.Conv2d(1, 1, kernel_size=(3, 2), padding=0, bias=False)
conv.weight.data = edge_kernel

maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

# Process each edge
fig, axes = plt.subplots(3, 3, figsize=(12, 10))

for idx, (edge, label) in enumerate([(edge_0, 'shift=0'), (edge_1, 'shift=1'), (edge_2, 'shift=2')]):
    edge_batch = edge.unsqueeze(0).unsqueeze(0)
    
    # Convolve
    conv_output = conv(edge_batch)
    
    # Pool
    pool_output = maxpool(conv_output)
    
    # Plot input
    axes[idx, 0].imshow(edge, cmap='gray')
    axes[idx, 0].set_title(f'Input ({label})', fontsize=12)
    axes[idx, 0].axis('off')
    
    # Plot after convolution
    axes[idx, 1].imshow(conv_output.squeeze().detach(), cmap='viridis')
    axes[idx, 1].set_title(f'After Conv', fontsize=12)
    axes[idx, 1].axis('off')
    
    # Plot after pooling
    axes[idx, 2].imshow(pool_output.squeeze().detach(), cmap='viridis')
    axes[idx, 2].set_title(f'After Pooling', fontsize=12)
    axes[idx, 2].axis('off')

plt.tight_layout()
plt.show()

print("\nObservation: Despite input shifts, the pooled outputs are very similar!")
print("This translation robustness is a key property of CNNs with pooling.")

## Part 6: Receptive Fields

The **receptive field** of a neuron is the region of the input that affects that neuron's activation.

**Key insight:** Pooling increases the receptive field of neurons in subsequent layers without adding parameters!

Let's trace how receptive fields grow through a simple CNN:

In [ ]:
# Calculate receptive field growth
def calculate_receptive_field(layers):
    """
    Calculate receptive field after a sequence of layers.
    layers: list of (type, kernel_size, stride) tuples
    """
    rf = 1  # Start with 1x1
    jump = 1  # How many input pixels one output pixel corresponds to
    
    print(f"Layer 0 (Input): RF = {rf}x{rf}, jump = {jump}")
    
    for i, (layer_type, kernel_size, stride) in enumerate(layers, 1):
        rf = rf + (kernel_size - 1) * jump
        jump = jump * stride
        print(f"Layer {i} ({layer_type} k={kernel_size}, s={stride}): RF = {rf}x{rf}, jump = {jump}")
    
    return rf

# Example CNN architecture
print("Example CNN Architecture:\n")
layers = [
    ('Conv', 3, 1),   # 3x3 conv, stride 1
    ('Conv', 3, 1),   # 3x3 conv, stride 1
    ('Pool', 2, 2),   # 2x2 max pool, stride 2
    ('Conv', 3, 1),   # 3x3 conv, stride 1
    ('Conv', 3, 1),   # 3x3 conv, stride 1
    ('Pool', 2, 2),   # 2x2 max pool, stride 2
]

final_rf = calculate_receptive_field(layers)

print(f"\nFinal receptive field: {final_rf}x{final_rf}")
print(f"\nInterpretation: Each neuron in the last layer 'sees' a {final_rf}x{final_rf} region of the input!")

### Visualizing Receptive Fields

Let's create a simple visualization showing how one output pixel relates to the input:

In [ ]:
# Visualize receptive field growth
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Layer 1: After first conv (3x3, RF=3)
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.invert_yaxis()
ax.set_aspect('equal')
ax.set_title('After Conv 3x3 (RF=3x3)', fontsize=14, fontweight='bold')
ax.grid(True)
# Draw input grid
for i in range(11):
    ax.axhline(i, color='lightgray', linewidth=0.5)
    ax.axvline(i, color='lightgray', linewidth=0.5)
# Highlight receptive field
rect = Rectangle((4, 4), 3, 3, fill=True, alpha=0.3, facecolor='red', edgecolor='red', linewidth=3)
ax.add_patch(rect)
ax.plot(5.5, 5.5, 'ro', markersize=10, label='Output pixel')
ax.set_xlabel('Input spatial dimension')
ax.legend()

# Layer 2: After pooling (2x2 stride 2, RF=4)
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.invert_yaxis()
ax.set_aspect('equal')
ax.set_title('After Conv + Pool 2x2 (RF=4x4)', fontsize=14, fontweight='bold')
ax.grid(True)
for i in range(11):
    ax.axhline(i, color='lightgray', linewidth=0.5)
    ax.axvline(i, color='lightgray', linewidth=0.5)
rect = Rectangle((3, 3), 4, 4, fill=True, alpha=0.3, facecolor='red', edgecolor='red', linewidth=3)
ax.add_patch(rect)
ax.plot(5, 5, 'ro', markersize=10, label='Output pixel')
ax.set_xlabel('Input spatial dimension')
ax.legend()

# Layer 3: After another conv (RF=8)
ax = axes[2]
ax.set_xlim(0, 16)
ax.set_ylim(0, 16)
ax.invert_yaxis()
ax.set_aspect('equal')
ax.set_title('After Conv + Pool + Conv (RF=8x8)', fontsize=14, fontweight='bold')
ax.grid(True)
for i in range(17):
    ax.axhline(i, color='lightgray', linewidth=0.5)
    ax.axvline(i, color='lightgray', linewidth=0.5)
rect = Rectangle((4, 4), 8, 8, fill=True, alpha=0.3, facecolor='red', edgecolor='red', linewidth=3)
ax.add_patch(rect)
ax.plot(8, 8, 'ro', markersize=10, label='Output pixel')
ax.set_xlabel('Input spatial dimension')
ax.legend()

plt.tight_layout()
plt.show()

print("The red region shows what part of the original input affects each output pixel.")
print("Notice how pooling dramatically expands the receptive field!")

## Part 7: Pooling with Different Strides

Like convolution, pooling has two key parameters:
- **Kernel size**: The size of the pooling window
- **Stride**: How much the window moves at each step

**Common configurations:**
- **Non-overlapping**: kernel_size = stride (e.g., 2x2 pool with stride 2)
- **Overlapping**: kernel_size > stride (e.g., 3x3 pool with stride 2)

Let's explore the difference:

In [ ]:
# Compare different pooling configurations
input_large = torch.randn(1, 1, 8, 8)

print("Input shape:", input_large.shape, "\n")

# Configuration 1: Non-overlapping (most common)
pool_2x2_s2 = nn.MaxPool2d(kernel_size=2, stride=2)
out1 = pool_2x2_s2(input_large)
print(f"Pool 2x2, stride 2 (non-overlapping): {out1.shape}")

# Configuration 2: Overlapping
pool_3x3_s2 = nn.MaxPool2d(kernel_size=3, stride=2)
out2 = pool_3x3_s2(input_large)
print(f"Pool 3x3, stride 2 (overlapping): {out2.shape}")

# Configuration 3: Stride 1 (minimal downsampling)
pool_2x2_s1 = nn.MaxPool2d(kernel_size=2, stride=1)
out3 = pool_2x2_s1(input_large)
print(f"Pool 2x2, stride 1 (minimal): {out3.shape}")

# Configuration 4: Larger kernel for aggressive downsampling
pool_4x4_s4 = nn.MaxPool2d(kernel_size=4, stride=4)
out4 = pool_4x4_s4(input_large)
print(f"Pool 4x4, stride 4 (aggressive): {out4.shape}")

### Output Size Formula for Pooling

The output size after pooling is calculated as:

$$\text{output\_size} = \left\lfloor \frac{\text{input\_size} - \text{kernel\_size}}{\text{stride}} \right\rfloor + 1$$

Let's verify this formula:

In [ ]:
# Verify pooling output size formula
def pool_output_size(input_size, kernel_size, stride):
    return (input_size - kernel_size) // stride + 1

input_size = 8
configs = [
    (2, 2),  # kernel, stride
    (3, 2),
    (2, 1),
    (4, 4),
]

print(f"Input size: {input_size}x{input_size}\n")

for kernel, stride in configs:
    calculated = pool_output_size(input_size, kernel, stride)
    
    # Verify with PyTorch
    pool = nn.MaxPool2d(kernel_size=kernel, stride=stride)
    test_input = torch.randn(1, 1, input_size, input_size)
    actual_output = pool(test_input)
    actual_size = actual_output.shape[2]
    
    match = "✓" if calculated == actual_size else "✗"
    print(f"Kernel {kernel}x{kernel}, stride {stride}:")
    print(f"  Calculated: {calculated}x{calculated}")
    print(f"  Actual: {actual_size}x{actual_size} {match}\n")

## Part 8: Global Pooling - From 2D to 1D

**Global pooling** reduces each feature map to a single value by pooling over the entire spatial dimensions.

**Types:**
- **Global Max Pooling**: Take the maximum value across the entire feature map
- **Global Average Pooling (GAP)**: Take the average value across the entire feature map

**Common use:** Replacing fully connected layers at the end of CNNs (reduces parameters dramatically!).

In [ ]:
# Demonstrate global pooling
# Simulate feature maps from a CNN: (batch=1, channels=3, height=4, width=4)
feature_maps = torch.tensor([
    # Channel 0
    [[1., 2., 3., 4.],
     [5., 6., 7., 8.],
     [9., 10., 11., 12.],
     [13., 14., 15., 16.]],
    
    # Channel 1
    [[16., 15., 14., 13.],
     [12., 11., 10., 9.],
     [8., 7., 6., 5.],
     [4., 3., 2., 1.]],
    
    # Channel 2
    [[1., 1., 1., 1.],
     [2., 2., 2., 2.],
     [3., 3., 3., 3.],
     [4., 4., 4., 4.]]
]).unsqueeze(0)  # Add batch dimension

print("Input feature maps shape:", feature_maps.shape)
print("  (batch=1, channels=3, height=4, width=4)\n")

# Global Max Pooling
global_max_pool = nn.AdaptiveMaxPool2d((1, 1))  # Pool to 1x1
global_max_output = global_max_pool(feature_maps)

print("Global Max Pooling output shape:", global_max_output.shape)
print("Values (one per channel):", global_max_output.squeeze())
print()

# Global Average Pooling
global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))  # Pool to 1x1
global_avg_output = global_avg_pool(feature_maps)

print("Global Average Pooling output shape:", global_avg_output.shape)
print("Values (one per channel):", global_avg_output.squeeze())
print()

# Verify manual calculation for channel 0
channel_0 = feature_maps[0, 0, :, :]
manual_max = channel_0.max()
manual_avg = channel_0.mean()

print(f"Manual verification for channel 0:")
print(f"  Max: {manual_max} (matches: {manual_max == global_max_output[0, 0, 0, 0]})")
print(f"  Avg: {manual_avg} (matches: {torch.isclose(manual_avg, global_avg_output[0, 0, 0, 0])})")

### Global Average Pooling vs Fully Connected Layer

Global Average Pooling is often used as a replacement for fully connected layers. Let's compare:

In [ ]:
# Compare GAP vs FC for classification
batch_size = 32
num_channels = 512
spatial_size = 7
num_classes = 1000

# Simulate CNN output before classification
cnn_output = torch.randn(batch_size, num_channels, spatial_size, spatial_size)

print(f"CNN output shape: {cnn_output.shape}")
print(f"  Batch: {batch_size}")
print(f"  Channels: {num_channels}")
print(f"  Spatial: {spatial_size}x{spatial_size}\n")

# Approach 1: Flatten + Fully Connected
flatten_size = num_channels * spatial_size * spatial_size
fc_params = flatten_size * num_classes

print("Approach 1: Fully Connected Layer")
print(f"  Flatten to: {flatten_size:,}")
print(f"  FC parameters: {fc_params:,}")
print(f"  Memory (FP32): {fc_params * 4 / (1024**2):.2f} MB\n")

# Approach 2: Global Average Pooling + Small FC
gap_params = num_channels * num_classes

print("Approach 2: Global Average Pooling + FC")
print(f"  GAP to: {num_channels}")
print(f"  FC parameters: {gap_params:,}")
print(f"  Memory (FP32): {gap_params * 4 / (1024**2):.2f} MB\n")

reduction = fc_params / gap_params
print(f"Parameter reduction: {reduction:.1f}x fewer parameters!")
print(f"\nThis is why modern CNNs (ResNet, EfficientNet, etc.) use GAP.")

### Implementing Global Average Pooling Manually

Let's see how GAP works under the hood:

In [ ]:
# Manual implementation of Global Average Pooling
test_input = torch.randn(2, 3, 4, 4)  # Batch=2, Channels=3, 4x4 spatial

# Method 1: Using AdaptiveAvgPool2d
gap_layer = nn.AdaptiveAvgPool2d((1, 1))
output_pytorch = gap_layer(test_input).squeeze(-1).squeeze(-1)  # Remove spatial dims

# Method 2: Manual implementation
output_manual = test_input.mean(dim=(2, 3))  # Average over height and width

# Method 3: Using mean directly on spatial dimensions
output_mean = torch.mean(test_input, dim=[2, 3])

print("PyTorch GAP output shape:", output_pytorch.shape)
print("Manual output shape:", output_manual.shape)
print("\nOutputs match:", torch.allclose(output_pytorch, output_manual))
print("All three methods match:", 
      torch.allclose(output_pytorch, output_manual) and 
      torch.allclose(output_manual, output_mean))

print("\nGlobal Average Pooling is simply: mean over spatial dimensions (H, W)")

## Part 9: Pooling in Real CNN Architectures

Let's build a simple CNN with pooling and analyze how dimensions change:

In [ ]:
# Build a simple CNN with pooling layers
class CNNWithPooling(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 32x32 -> 32x32
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)       # 32x32 -> 16x16
        
        # Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # 16x16 -> 16x16
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)       # 16x16 -> 8x8
        
        # Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1) # 8x8 -> 8x8
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)        # 8x8 -> 4x4
        
        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))           # 4x4 -> 1x1
        
        # Classification head
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # Block 1
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        print(f"After block 1: {x.shape}")
        
        # Block 2
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        print(f"After block 2: {x.shape}")
        
        # Block 3
        x = F.relu(self.conv3(x))
        x = self.pool3(x)
        print(f"After block 3: {x.shape}")
        
        # Global pooling
        x = self.global_pool(x)
        print(f"After global pool: {x.shape}")
        
        # Flatten and classify
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        print(f"After FC: {x.shape}")
        
        return x

# Test the model
model = CNNWithPooling(num_classes=10)
dummy_input = torch.randn(4, 3, 32, 32)  # Batch of 4 RGB images, 32x32

print("Input shape:", dummy_input.shape)
print("\nForward pass shape progression:")
output = model(dummy_input)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

### Analyzing Parameter Distribution

Let's see where parameters are concentrated in this architecture:

In [ ]:
# Analyze parameter distribution
print("Layer-wise parameter count:\n")
for name, param in model.named_parameters():
    print(f"{name:20s}: {param.numel():>8,} parameters, shape {list(param.shape)}")

# Compare with a model WITHOUT global pooling
fc_without_gap_params = 128 * 4 * 4 * 10  # 128 channels, 4x4 spatial, 10 classes
fc_with_gap_params = 128 * 10

print(f"\nFinal FC layer comparison:")
print(f"  Without GAP: {fc_without_gap_params:,} parameters")
print(f"  With GAP: {fc_with_gap_params:,} parameters")
print(f"  Reduction: {fc_without_gap_params / fc_with_gap_params:.1f}x")

print("\n✓ Pooling layers have ZERO learnable parameters!")
print("  They reduce spatial dimensions without adding model complexity.")

## Part 10: Pooling Trade-offs and Design Choices

Let's explore when to use each type of pooling and common design patterns.

### Max Pooling vs Average Pooling: When to Use Each?

**Max Pooling:**
- ✓ Preserves strongest features (e.g., edges, textures)
- ✓ More common in practice
- ✓ Better for detecting presence of features
- ✗ Discards weaker signals that might be useful

**Average Pooling:**
- ✓ Smoother, considers all values
- ✓ Better for distributed features
- ✓ Provides smoother gradients
- ✗ Can dilute strong signals

Let's demonstrate the difference with a concrete example:

In [ ]:
# Create two test patterns
# Pattern 1: Sparse strong activations (peaks)
sparse_pattern = torch.tensor([
    [0., 0., 0., 0.],
    [0., 9., 0., 0.],
    [0., 0., 0., 0.],
    [0., 0., 0., 8.]
])

# Pattern 2: Distributed moderate activations
distributed_pattern = torch.tensor([
    [3., 3., 2., 2.],
    [3., 3., 2., 2.],
    [2., 2., 4., 4.],
    [2., 2., 4., 4.]
])

# Apply both pooling types
maxpool = nn.MaxPool2d(2, 2)
avgpool = nn.AvgPool2d(2, 2)

# Process sparse pattern
sparse_batch = sparse_pattern.unsqueeze(0).unsqueeze(0)
sparse_max = maxpool(sparse_batch).squeeze()
sparse_avg = avgpool(sparse_batch).squeeze()

# Process distributed pattern
dist_batch = distributed_pattern.unsqueeze(0).unsqueeze(0)
dist_max = maxpool(dist_batch).squeeze()
dist_avg = avgpool(dist_batch).squeeze()

# Display results
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Row 1: Sparse pattern
axes[0, 0].imshow(sparse_pattern, cmap='viridis', vmin=0, vmax=10)
axes[0, 0].set_title('Sparse Pattern (Input)', fontsize=12, fontweight='bold')
axes[0, 1].imshow(sparse_max, cmap='viridis', vmin=0, vmax=10)
axes[0, 1].set_title('Max Pooling', fontsize=12, fontweight='bold')
axes[0, 2].imshow(sparse_avg, cmap='viridis', vmin=0, vmax=10)
axes[0, 2].set_title('Avg Pooling', fontsize=12, fontweight='bold')

# Row 2: Distributed pattern
axes[1, 0].imshow(distributed_pattern, cmap='viridis', vmin=0, vmax=10)
axes[1, 0].set_title('Distributed Pattern (Input)', fontsize=12, fontweight='bold')
axes[1, 1].imshow(dist_max, cmap='viridis', vmin=0, vmax=10)
axes[1, 1].set_title('Max Pooling', fontsize=12, fontweight='bold')
axes[1, 2].imshow(dist_avg, cmap='viridis', vmin=0, vmax=10)
axes[1, 2].set_title('Avg Pooling', fontsize=12, fontweight='bold')

for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Sparse pattern:")
print(f"  Max pooling: {sparse_max.flatten()}  (preserves peaks!)")
print(f"  Avg pooling: {sparse_avg.flatten()}  (diluted)\n")

print("Distributed pattern:")
print(f"  Max pooling: {dist_max.flatten()}")
print(f"  Avg pooling: {dist_avg.flatten()}  (better represents the distribution)\n")

print("Conclusion: Max pooling is better for sparse, strong features.")
print("             Average pooling is better for distributed, moderate features.")

## Part 11: Alternatives to Pooling

Modern architectures sometimes avoid pooling entirely. Let's explore alternatives:

**1. Strided Convolutions:**
- Use conv layers with stride > 1 instead of pooling
- Learnable downsampling (parameters can be trained)
- Used in: ResNet variants, some GANs

**2. Dilated Convolutions:**
- Increase receptive field without pooling
- Used in: Semantic segmentation, audio models

In [ ]:
# Compare pooling vs strided convolution
input_test = torch.randn(1, 32, 16, 16)

print("Input shape:", input_test.shape, "\n")

# Method 1: Convolution + Max Pooling
conv1 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
pool = nn.MaxPool2d(kernel_size=2, stride=2)
output1 = pool(conv1(input_test))

# Method 2: Strided Convolution
conv_strided = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
output2 = conv_strided(input_test)

print("Method 1 (Conv + Pool):")
print(f"  Output shape: {output1.shape}")
print(f"  Parameters: {sum(p.numel() for p in [conv1.weight, conv1.bias])}")
print(f"  Pooling params: 0 (pooling has no parameters)\n")

print("Method 2 (Strided Conv):")
print(f"  Output shape: {output2.shape}")
print(f"  Parameters: {sum(p.numel() for p in [conv_strided.weight, conv_strided.bias])}\n")

print("Key differences:")
print("  • Both reduce spatial dimensions by 2x")
print("  • Strided conv is learnable (can adapt downsampling)")
print("  • Pooling is fixed (no learning, but simpler)")
print("  • Strided conv uses same # of parameters as regular conv")

## Part 12: Practical Considerations

Let's discuss common pitfalls and best practices when using pooling:

### Pitfall 1: Losing Too Much Information

Aggressive pooling can destroy important details, especially early in the network.

In [ ]:
# Demonstrate information loss from aggressive pooling
# Create a detailed pattern
detailed_pattern = torch.zeros(8, 8)
detailed_pattern[2:6, 2:6] = 1.0
detailed_pattern[3:5, 3:5] = 0.5

# Apply different pooling strategies
input_batch = detailed_pattern.unsqueeze(0).unsqueeze(0)

pool_2x2 = nn.MaxPool2d(2, 2)
pool_4x4 = nn.MaxPool2d(4, 4)
pool_8x8 = nn.MaxPool2d(8, 8)

output_2x2 = pool_2x2(input_batch).squeeze()
output_4x4 = pool_4x4(input_batch).squeeze()
output_8x8 = pool_8x8(input_batch).squeeze()

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(detailed_pattern, cmap='gray', interpolation='nearest')
axes[0].set_title('Original (8x8)', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(output_2x2, cmap='gray', interpolation='nearest')
axes[1].set_title('After 2x2 pool (4x4)', fontsize=14, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(output_4x4, cmap='gray', interpolation='nearest')
axes[2].set_title('After 4x4 pool (2x2)', fontsize=14, fontweight='bold')
axes[2].axis('off')

# For 8x8 pool, the output is a scalar, so we display it as text
axes[3].text(0.5, 0.5, f'{output_8x8.item():.2f}', 
            ha='center', va='center', fontsize=48, fontweight='bold')
axes[3].set_title('After 8x8 pool (1x1)', fontsize=14, fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("Notice: Aggressive pooling (4x4, 8x8) destroys spatial structure!")
print("Best practice: Use gradual pooling (2x2) multiple times instead.")

### Pitfall 2: Pooling Too Early

Pooling immediately after the first convolution can lose low-level details (edges, textures).

In [ ]:
# Compare architectures with early vs late pooling
class EarlyPooling(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)  # Pool immediately
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))  # 32x32
        x = self.pool1(x)          # 16x16 (early pooling)
        x = F.relu(self.conv2(x))  # 16x16
        x = self.pool2(x)          # 8x8
        return x

class LaterPooling(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv1b = nn.Conv2d(32, 32, 3, padding=1)  # Extra conv at full resolution
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))   # 32x32
        x = F.relu(self.conv1b(x))  # 32x32 (process more at full resolution)
        x = self.pool1(x)           # 16x16
        x = F.relu(self.conv2(x))   # 16x16
        x = self.pool2(x)           # 8x8
        return x

early = EarlyPooling()
later = LaterPooling()

test_input = torch.randn(1, 3, 32, 32)

print("Early Pooling Architecture:")
print(early)
print(f"\nLater Pooling Architecture:")
print(later)

print("\n💡 Best practice: Apply 2-3 conv layers at full resolution")
print("   before first pooling to extract rich low-level features.")

## Part 13: Pooling in Famous Architectures

Let's see how pooling is used in well-known CNN architectures:

**AlexNet (2012):**
- Max pooling after every 1-2 conv layers
- 3x3 pooling with stride 2 (overlapping)

**VGGNet (2014):**
- Max pooling after every 2-3 conv layers
- 2x2 pooling with stride 2 (non-overlapping)
- Very regular architecture

**ResNet (2015):**
- Initial 7x7 conv + max pool
- Strided convolutions for downsampling (instead of pooling)
- Global average pooling at the end

**EfficientNet (2019):**
- Mostly strided convolutions
- Global average pooling at the end
- No explicit max pooling layers

In [ ]:
# Simplified VGG-style architecture
class VGGStyle(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Block 1: 2 convs + pool
        self.conv1_1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Block 2: 2 convs + pool
        self.conv2_1 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Global average pooling
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        
        # Classifier
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # Block 1
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.pool1(x)
        
        # Block 2
        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = self.pool2(x)
        
        # Global pooling
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        
        return x

vgg_model = VGGStyle(num_classes=10)
print("VGG-Style Architecture:")
print(vgg_model)
print(f"\nTotal parameters: {sum(p.numel() for p in vgg_model.parameters()):,}")

# Test forward pass
test_input = torch.randn(1, 3, 32, 32)
output = vgg_model(test_input)
print(f"\nOutput shape: {output.shape}")

## Part 14: Key Takeaways

Let's summarize the essential concepts about pooling operations:

### What is Pooling?
- **Downsampling operation** that reduces spatial dimensions
- Aggregates values within a sliding window
- **Zero learnable parameters** (except adaptive pooling in some contexts)

### Types of Pooling:

**Max Pooling:**
- Takes maximum value in each window
- Most common in practice
- Good for sparse, strong features

**Average Pooling:**
- Takes mean value in each window
- Smoother, considers all values
- Good for distributed features

**Global Pooling:**
- Pools entire spatial dimensions to 1x1
- Replaces fully connected layers
- Dramatically reduces parameters

### Why Use Pooling?

1. **Computational Efficiency:**
   - Reduces feature map size
   - Decreases memory and computation
   - Reduces parameters in subsequent layers

2. **Translation Invariance:**
   - Makes network robust to small shifts
   - Aggregates nearby features
   - Important for image classification

3. **Receptive Field Expansion:**
   - Each layer sees larger input region
   - Builds hierarchical features
   - No additional parameters needed

### Best Practices:

✓ Use 2x2 max pooling with stride 2 (most common)

✓ Apply 2-3 conv layers before first pooling

✓ Use global average pooling before classification

✓ Consider strided convolutions as alternative

✗ Avoid aggressive pooling (4x4, 8x8)

✗ Don't pool immediately after first conv

✗ Don't over-pool (losing too much information)

### Common Configurations:

```python
# Standard max pooling
nn.MaxPool2d(kernel_size=2, stride=2)

# Overlapping pooling (AlexNet)
nn.MaxPool2d(kernel_size=3, stride=2)

# Global average pooling
nn.AdaptiveAvgPool2d((1, 1))

# Strided convolution alternative
nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1)
```

## Part 15: Exercises for Practice

Try these exercises to deepen your understanding:

**Exercise 1:** Create a 6x6 input and apply max pooling with:
- 2x2 kernel, stride 2
- 3x3 kernel, stride 1
- 3x3 kernel, stride 2

Calculate expected output sizes and verify with PyTorch.

**Exercise 2:** Build a CNN with 3 blocks (conv-pool) and calculate:
- Receptive field at each layer
- Memory usage at each layer
- Total parameter count

**Exercise 3:** Implement a custom pooling operation that takes the **median** value in each window (median pooling).

**Exercise 4:** Compare a CNN with max pooling vs strided convolutions:
- Train both on MNIST
- Compare accuracy, speed, and parameter count

**Exercise 5:** Visualize receptive fields:
- Build a 4-layer CNN
- Calculate theoretical receptive field
- Use gradient visualization to confirm

**Exercise 6:** Experiment with global pooling:
- Replace FC layers in a small CNN with GAP
- Measure parameter reduction
- Compare performance on CIFAR-10

Good luck! 🚀